# Machine Translation

Translation is the task that paid for NLP research for thrity years and keeps paying now.

## Problem Definition

No word level alignment survives in translation.

Matchine translation is the task that forces NLP to invent encoder-decoders, attention, transformers,
and eventually the whole LLM paradigm.

## Basic Concept

### MT Pipeline

Tokenize --> Encode --> Decode with attention --> Detokenize

The encoder reads the source in its language's tokenization. The decoder generates the target,
on subword at a time, using the encoder's output via cross-attention.

Decoding uses **beam search** to avoid the greedy-decoding trap.

# Build your Own

## Load Pretained MT

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"

tok = AutoTokenizer.from_pretrained(model_id, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

src = "The cats are running."
inputs = tok(src, return_tensors="pt")

out = model.generate(
    **inputs,
    forced_bos_token_id=tok.convert_tokens_to_ids("fra_Latn"),
    num_beams=5,
    length_penalty=1.0,
    max_new_tokens=64
)

print(tok.batch_decode(out, skip_special_tokens=True)[0])

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Les chats courent.


## BLEU and chrF

In [9]:
import sacrebleu

hypothess = ["The cat is running."]
references = [["The cats are running."], ["The cat were running."]]

bleu = sacrebleu.corpus_bleu(hypothess, references)
chrf = sacrebleu.corpus_chrf(hypothess, references)

print(f"BLEU: {bleu.score}, chrF: {chrf.score}")

BLEU: 30.213753973567677, chrF: 58.71494455010393


## Fine-tuning for a domain

In [ ]:
import torch
from transformers import Trainer, TrainingArguments
from datasets import Dataset

pairs = [
    {"src":"What the fuck.", "tgt": "卧槽。"}
]

ds = Dataset.from_list(pairs)

def preprocess(ex):
    return tok(
        ex["src"],
        text_target=ex["tgt"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

ds = ds.map(preprocess, remove_columns=["src", "tgt"])

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=3e-5
)

Trainer(model=model, args=args, train_dataset=ds).train()


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`